# **I. Environment Setup on Google Colab**

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
# # Install necessary libraries
# !pip install torch==2.2.0 torchvision==0.17.0
# !pip install retina-face
# !pip install gbc-face==2025.1.0 albumentations==1.3.1
# !pip install pytorch-lightning==2.2.0 torchmetrics==1.4.0
# !pip install pandas scikit-learn opencv-python


# **4. Model Definition and Feature Extraction**

In [ ]:
!git clone https://github.com/aliyun/GB-CosFace.git
%cd GB-CosFace

Cloning into 'GB-CosFace'...
remote: Enumerating objects: 42, done.
remote: Counting objects: 100% (3/3), done.
remote: Compressing objects: 100% (3/3), done.
remote: Total 42 (delta 0), reused 0 (delta 0), pack-reused 39 (from 1)
Receiving objects: 100% (42/42), 29.45 KiB | 29.45 MiB/s, done.
Resolving deltas: 100% (3/3), done.
/content/GB-CosFace


In [ ]:
!pip insta
ll -r requirements.txt
!pip install timm

SyntaxError: invalid syntax (ipython-input-3-394152302.py, line 2)

In [ ]:
# !wget -O /content/drive/MyDrive/PFE/models/gb_cosface_backbone.pth http://virutalbuy-public.oss-cn-hangzhou.aliyuncs.com/share/GB-CosFace/models/backbone_paper.pth

## **I- Original Work**

### ***1. Pure Inference :***

In [ ]:
# ------------------ Imports ------------------
import torch
from torchvision import transforms
from PIL import Image
import os
import numpy as np
from backbone.backbone_def import BackboneFactory
from tqdm import tqdm

# ------------------ Device ------------------
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# ------------------ Paths ------------------
PRETRAINED_MODEL_PATH = "/content/drive/MyDrive/PFE/models/gb_cosface_backbone.pth"
BACKBONE_CONF_PATH = "/content/GB-CosFace/configs/backbone_conf.yaml"

SPLIT_BASE = "/content/drive/MyDrive/PFE/datasets/dataset_split_job2"
EMB_SAVE_DIR = "/content/drive/MyDrive/PFE/embeddings_job2_test"
os.makedirs(EMB_SAVE_DIR, exist_ok=True)

# ------------------ Load Pretrained Model ------------------
def load_iresnet100(path):
    backbone_type = "iresnet100"
    backbone_factory = BackboneFactory(backbone_type, BACKBONE_CONF_PATH)
    backbone = backbone_factory.get_backbone()
    state_dict = torch.load(path, map_location=DEVICE)
    backbone.load_state_dict(state_dict)
    backbone.to(DEVICE)
    backbone.eval()
    return backbone

model = load_iresnet100(PRETRAINED_MODEL_PATH)

# ------------------ Image Transform ------------------
transform = transforms.Compose([
    transforms.Resize((112, 112)),
    transforms.ToTensor(),
    transforms.Normalize([0.5], [0.5])
])

# ------------------ Embedding Extraction ------------------
def extract_embedding(image_path, model):
    image = Image.open(image_path).convert('RGB')
    img_t = transform(image).unsqueeze(0).to(DEVICE)
    with torch.no_grad():
        emb = model(img_t).cpu().numpy().flatten()
        emb = emb / np.linalg.norm(emb)
    return emb

# ------------------ Save Embeddings with IDs ------------------
def process_split(split_name):
    folder_path = os.path.join(SPLIT_BASE, split_name)

    # ✅ Fix file names by replacing "/" with "_"
    safe_split_name = split_name.replace("/", "_")
    save_path = os.path.join(EMB_SAVE_DIR, f"{safe_split_name}_embeddings.npy")
    save_ids_path = os.path.join(EMB_SAVE_DIR, f"{safe_split_name}_labels.npy")

    embeddings = []
    labels = []

    for identity in tqdm(sorted(os.listdir(folder_path)), desc=f"Processing {split_name}"):
        identity_folder = os.path.join(folder_path, identity)
        if not os.path.isdir(identity_folder):
            continue
        for img_file in os.listdir(identity_folder):
            if img_file.lower().endswith(('.jpg', '.jpeg', '.png')):
                img_path = os.path.join(identity_folder, img_file)
                try:
                    emb = extract_embedding(img_path, model)
                    embeddings.append(emb)
                    labels.append(identity if split_name != 'test/unknown' else 'unknown')
                except Exception as e:
                    print(f"❌ Error on {img_path}: {e}")

    np.save(save_path, np.array(embeddings))
    np.save(save_ids_path, np.array(labels))
    print(f"✅ Saved {safe_split_name} embeddings and labels!")

# ------------------ Run for all splits ------------------
for split in ['gallery', 'test/known', 'test/unknown']:
    process_split(split)


backbone param:
{'params': None}


Processing gallery: 100%|██████████| 119/119 [00:03<00:00, 39.32it/s]


✅ Saved gallery embeddings and labels!


Processing test/known: 100%|██████████| 119/119 [00:54<00:00,  2.17it/s]


✅ Saved test_known embeddings and labels!


Processing test/unknown: 100%|██████████| 20/20 [00:42<00:00,  2.11s/it]

✅ Saved test_unknown embeddings and labels!


In [ ]:
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity

# ------------------ Paths ------------------
EMB_SAVE_DIR = "/content/drive/MyDrive/PFE/embeddings_job2_test"

# ------------------ Load embeddings ------------------
gallery_emb = np.load(f"{EMB_SAVE_DIR}/gallery_embeddings.npy")
gallery_ids = np.load(f"{EMB_SAVE_DIR}/gallery_labels.npy", allow_pickle=True)

probe_known_emb = np.load(f"{EMB_SAVE_DIR}/test_known_embeddings.npy")
probe_known_ids = np.load(f"{EMB_SAVE_DIR}/test_known_labels.npy", allow_pickle=True)

probe_unknown_emb = np.load(f"{EMB_SAVE_DIR}/test_unknown_embeddings.npy")

# ------------------ Compute similarity scores ------------------
def match_probe_to_gallery(probe_embeddings, gallery_embeddings):
    return cosine_similarity(probe_embeddings, gallery_embeddings)  # shape (num_probes, num_gallery)

# Known probe: similarity & prediction
known_sim = match_probe_to_gallery(probe_known_emb, gallery_emb)
top_scores_known = np.max(known_sim, axis=1)
top_preds_known = gallery_ids[np.argmax(known_sim, axis=1)]

# Unknown probe: only scores
unknown_sim = match_probe_to_gallery(probe_unknown_emb, gallery_emb)
top_scores_unknown = np.max(unknown_sim, axis=1)

# ------------------ CMC Rank-1 ------------------
rank1_correct = np.sum(probe_known_ids == top_preds_known)
rank1_total = len(probe_known_ids)
rank1_accuracy = rank1_correct / rank1_total
print(f"\n🎯 Rank-1 Accuracy (Known Probes): {rank1_accuracy:.4f}")

# ------------------ TPIR@FPIR Evaluation ------------------
def compute_tpir_fpirs(tp_scores, fp_scores, fpir_targets):
    thresholds = np.sort(fp_scores)[::-1]  # Descending
    n_fp_total = len(fp_scores)
    n_tp_total = len(tp_scores)

    results = {}

    for fpir_target in fpir_targets:
        max_fp = int(np.floor(fpir_target * n_fp_total))
        if max_fp == 0:
            max_fp = 1

        thresh_idx = max_fp - 1
        if thresh_idx >= len(thresholds):
            thresh_idx = len(thresholds) - 1

        threshold = thresholds[thresh_idx]

        tp = np.sum(tp_scores >= threshold)
        tpir = tp / n_tp_total
        results[fpir_target] = {
            "threshold": float(threshold),
            "TPIR": float(tpir)
        }

    return results

# Compute metrics
fpir_targets = [1e-1, 1e-2, 1e-3, 1e-4]
results = compute_tpir_fpirs(top_scores_known, top_scores_unknown, fpir_targets)

# ------------------ Report ------------------
print("\n📊 TPIR @ FPIR Results:")
for fpir, res in results.items():
    print(f"  TPIR @ FPIR={fpir:.0e}: {res['TPIR']:.4f} (Threshold: {res['threshold']:.4f})")



🎯 Rank-1 Accuracy (Known Probes): 0.9367

📊 TPIR @ FPIR Results:
  TPIR @ FPIR=1e-01: 0.9419 (Threshold: 0.2597)
  TPIR @ FPIR=1e-02: 0.8690 (Threshold: 0.3738)
  TPIR @ FPIR=1e-03: 0.5769 (Threshold: 0.5910)
  TPIR @ FPIR=1e-04: 0.5769 (Threshold: 0.5910)
